# Cell Type Visualization
*Bladder DIVE — CellDIVE Spatial Proteomics*

Visualises the annotated cell populations in **`celldive_protein_matrix_celltypes.h5ad`**.

### Pipeline position
```
determine_binary_threshold → build_protein_cell_matrix → analyze_cell_types → [YOU ARE HERE]
```

### What this notebook produces
| Section | Output |
|---------|--------|
| 1. QC summary | Artifact / clean counts, bar chart |
| 2. Scanpy preprocessing | PCA, UMAP embedding on **clean** cells |
| 3. UMAP | Coloured by cell type, QC label, individual markers |
| 4. Spatial scatter | Cell centroids coloured by cell type on tissue |
| 5. Marker heatmap | Mean z-scored expression per cell type |
| 6. Dot plot | % expressing + mean expression per cell type per marker |
| 7. Spatial co-occurrence | Which cell types are spatially adjacent? |
| 8. QC spatial map | Artefact locations on tissue |


In [1]:
import matplotlib
matplotlib.use("Agg")  # headless backend for nbconvert; remove if running interactively
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import anndata as ad
import scanpy as sc
from scipy.spatial import cKDTree

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)
plt.rcParams["figure.dpi"] = 100

try:
    import squidpy as sq
    HAS_SQUIDPY = True
    print(f"squidpy {sq.__version__} available")
except ImportError:
    HAS_SQUIDPY = False
    print("squidpy not installed — spatial co-occurrence will use a manual approach")


squidpy 1.6.6 available


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CELLTYPES_H5AD = PROJECT_ROOT / "output" / "celldive_protein_matrix_celltypes.h5ad"
FIG_DIR = PROJECT_ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
# DPI for static PNGs (UMAP, heatmaps, etc.). Spatial scatter also uses SPATIAL_DPI in its cell.
FIG_DPI = 300

# Colours: red family for artefacts/unassigned, blue family for real cell types
ARTIFACT_COLOUR = "#d73027"
UNASSIGNED_COLOUR = "#fc8d59"

print(f"H5AD   : {CELLTYPES_H5AD}")
print(f"Figures: {FIG_DIR}")


H5AD   : /home/steve/Projects/HeLab/BladderDIVE/output/celldive_protein_matrix_celltypes.h5ad
Figures: /home/steve/Projects/HeLab/BladderDIVE/output/figures


In [3]:
adata = ad.read_h5ad(CELLTYPES_H5AD)
print(adata)
print()
print("obs columns :", list(adata.obs.columns))
print("layers      :", list(adata.layers.keys()))
print("uns keys    :", list(adata.uns.keys()))


AnnData object with n_obs × n_vars = 926006 × 23
    obs: 'cell_id', 'area', 'centroid_x', 'centroid_y', 'qc_label', 'is_clean', 'cell_type'
    uns: 'background_estimate', 'background_percentile', 'binary_threshold', 'binary_thresholds', 'cell_type_counts', 'cell_type_rules', 'data_source', 'qc_gates', 'sample_id'
    obsm: 'X_umap_clean', 'spatial'
    layers: 'binary'

obs columns : ['cell_id', 'area', 'centroid_x', 'centroid_y', 'qc_label', 'is_clean', 'cell_type']
layers      : ['binary']
uns keys    : ['background_estimate', 'background_percentile', 'binary_threshold', 'binary_thresholds', 'cell_type_counts', 'cell_type_rules', 'data_source', 'qc_gates', 'sample_id']


## 1. QC Summary

In [4]:
ct_counts = adata.obs["cell_type"].value_counts()
qc_counts = adata.obs["qc_label"].value_counts()

n_total       = len(adata)
n_clean       = int(adata.obs["is_clean"].sum())
n_no_nucleus  = int((adata.obs["qc_label"] == "Artifact_NoNucleus").sum())
n_spillover   = int((adata.obs["qc_label"] == "Artifact_Spillover").sum())

print(f"Total cells          : {n_total:,}")
print(f"Clean (pass QC)      : {n_clean:,}  ({100*n_clean/n_total:.1f} %)")
print(f"Artifact_NoNucleus   : {n_no_nucleus:,}  ({100*n_no_nucleus/n_total:.1f} %)")
print(f"Artifact_Spillover   : {n_spillover:,}  ({100*n_spillover/n_total:.1f} %)")
print()
print("Cell type breakdown (all cells):")
print(ct_counts.to_string())


Total cells          : 926,006
Clean (pass QC)      : 617,128  (66.6 %)
Artifact_NoNucleus   : 231,502  (25.0 %)
Artifact_Spillover   : 77,376  (8.4 %)

Cell type breakdown (all cells):
cell_type
Myofibroblast         232480
Artifact_NoNucleus    231502
Unassigned             80964
Artifact_Spillover     77376
Collagen_stroma        69013
Mesenchymal            57863
Epithelial_EPCAM       48174
Fibroblast             39743
Endothelial            39005
CD8_T                  27651
CD4_T                   5264
B_cell                  4971
T_cell                  3993
Epithelial              3289
Dendritic               1557
Macrophage              1391
Other_immune             541
Monocyte                 503
M2_like                  386
NK_cell                  340


In [5]:
# Bar chart of cell type counts
fig, ax = plt.subplots(figsize=(10, 5))
sorted_ct = ct_counts.sort_values(ascending=True)
colours = [
    ARTIFACT_COLOUR if ("Artifact" in ct or ct == "Unassigned") else "steelblue"
    for ct in sorted_ct.index
]
bars = ax.barh(sorted_ct.index, sorted_ct.values, color=colours)
ax.set_xlabel("Cell count")
ax.set_title(f"Cell type counts — {n_total:,} total cells")
for bar, val in zip(bars, sorted_ct.values):
    ax.text(bar.get_width() + n_total * 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "celltype_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Scanpy Preprocessing (UMAP)

UMAP is computed **only on clean cells** (artefacts excluded). Steps:
1. `sc.pp.log1p` — log-transform continuous intensities
2. `sc.pp.scale` — z-score each marker
3. `sc.pp.pca` — reduce to 15 PCs
4. `sc.pp.neighbors` — k-NN graph in PCA space
5. `sc.tl.umap` — 2-D embedding


In [6]:
# Work on clean cells only for UMAP
adata_clean = adata[adata.obs["is_clean"]].copy()
print(f"Clean cells for UMAP: {adata_clean.n_obs:,}")

# Exclude DAPI channels from the expression matrix used for UMAP
dapi_mask = ~adata_clean.var_names.str.upper().str.startswith("DAPI")
adata_clean = adata_clean[:, dapi_mask].copy()
print(f"Markers used (excluding DAPI): {list(adata_clean.var_names)}")


Clean cells for UMAP: 617,128
Markers used (excluding DAPI): ['CD45', 'CD3E', 'Ki67', 'CD8a', 'VIM', 'CD68', 'HLADR', 'CD31', 'ACTA2', 'CD20', 'CD163', 'CD44', 'PANCK', 'CD38', 'CD11c', 'PDGFRA', 'COL1A1', 'CD14', 'EPCAM', 'CD56', 'CD45RO']


In [7]:
# Log-transform, scale, PCA
sc.pp.log1p(adata_clean)
sc.pp.scale(adata_clean, max_value=10)
sc.pp.pca(adata_clean, n_comps=15, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata_clean, n_pcs=15, log=True)


In [8]:
# Build kNN graph and UMAP
sc.pp.neighbors(adata_clean, n_neighbors=15, n_pcs=10)
sc.tl.umap(adata_clean)
print("UMAP done.")


UMAP done.


## 3. UMAP

PCA → neighbors → UMAP on **clean** cells only (scaled log1p protein space).

Exports **`umap_cell_type.png`** (`legend_loc="on data"`, small font) and **`umap_cell_type_legend_right.png`**
(`legend_loc="right margin"` — no overlapping labels). Both use **`FIG_DPI`** (default 300) in the config cell.

In [9]:
# Cell type UMAP — `legend_loc="on data"` places names at cluster medians; they overlap when clusters overlap.
# We save two versions: compact on-data labels + a side legend with no overlap.

_umap_kw = dict(
    color="cell_type",
    title="UMAP — cell type (clean cells)",
    palette="tab20",
    size=2,
    legend_fontweight="normal",
    legend_fontoutline=1,
)

sc.pl.umap(adata_clean, legend_loc="on data", legend_fontsize=5, **_umap_kw, save=False)
plt.savefig(FIG_DIR / "umap_cell_type.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

sc.pl.umap(adata_clean, legend_loc="right margin", legend_fontsize=8, **_umap_kw, save=False)
plt.savefig(FIG_DIR / "umap_cell_type_legend_right.png", dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / 'umap_cell_type.png'} (on-data labels, small font)")
print(f"Saved → {FIG_DIR / 'umap_cell_type_legend_right.png'} (legend in right margin, no overlap)")


In [10]:
# Individual marker UMAPs — show all available markers in a grid
markers = list(adata_clean.var_names)
ncols = 4
nrows = int(np.ceil(len(markers) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.ravel()

sc.pl.umap(
    adata_clean,
    color=markers,
    ncols=ncols,
    use_raw=False,
    show=False,
    save=False,
)
plt.suptitle("UMAP — per-marker expression (z-scored)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "umap_markers.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Spatial Scatter

Each dot is one cell centroid, coloured by cell type. Exports:

- `spatial_cell_types.png` — may **subsample** (`MAX_PLOT_CELLTYPE`) for speed; **`SPATIAL_DPI`** (default 300) controls sharpness.
- `spatial_qc.png` — combined Pass / Artifact_NoNucleus / Artifact_Spillover.
- **`spatial_qc_layer_<label>.png`** — **one file per QC class** with identical axis limits, so you can inspect artefacts alone or stack in an image editor. Set **`PLOT_QC_ALL_CELLS = True`** (default) to use **every cell** on QC maps (no subsample).

For **true layer on/off** and zoom, use **Napari** (`napari_load_image_adata`, napari-env).


In [11]:
# Assign a colour to every cell type (consistent with UMAP palette)
cell_types = list(adata.obs["cell_type"].cat.categories)
palette = plt.get_cmap("tab20").colors
ct_colour = {}
real_idx = 0
for ct in cell_types:
    if "Artifact" in ct:
        ct_colour[ct] = ARTIFACT_COLOUR
    elif ct == "Unassigned":
        ct_colour[ct] = UNASSIGNED_COLOUR
    else:
        ct_colour[ct] = palette[real_idx % len(palette)]
        real_idx += 1

colours_per_cell = [ct_colour[ct] for ct in adata.obs["cell_type"]]


In [12]:
# Spatial scatter — cell types (subsampled for speed) + QC (full or subsampled) + per-layer PNGs
from matplotlib.lines import Line2D

SPATIAL_DPI = 300  # raise for publication; file size grows
MAX_PLOT_CELLTYPE = 300_000  # subsample for the busy cell-type map only
PLOT_QC_ALL_CELLS = True     # True = use every cell for QC plots (slower, no random subsample)

xy = np.asarray(adata.obsm["spatial"], dtype=np.float64)
n = len(adata)
xmin, xmax = xy[:, 0].min(), xy[:, 0].max()
ymin, ymax = xy[:, 1].min(), xy[:, 1].max()
pad = 0.01 * max(xmax - xmin, ymax - ymin)

# --- Cell-type map (subsample if huge) ---
if n > MAX_PLOT_CELLTYPE:
    rng = np.random.default_rng(42)
    idx_ct = np.sort(rng.choice(n, MAX_PLOT_CELLTYPE, replace=False))
    xy_plot_ct = xy[idx_ct]
    col_plot_ct = [colours_per_cell[i] for i in idx_ct]
    print(f"Cell-type map: {MAX_PLOT_CELLTYPE:,} random cells (of {n:,})")
else:
    idx_ct = None
    xy_plot_ct = xy
    col_plot_ct = colours_per_cell

fig, ax = plt.subplots(figsize=(14, 14))
ax.scatter(xy_plot_ct[:, 0], xy_plot_ct[:, 1], c=col_plot_ct, s=0.35, alpha=0.65,
           linewidths=0, rasterized=True)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.axis("off")
ax.set_title("Spatial map — cell types", fontsize=14)
ax.set_xlim(xmin - pad, xmax + pad)
ax.set_ylim(ymax + pad, ymin - pad)

handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=ct_colour[ct],
           markersize=8, label=ct)
    for ct in sorted(ct_colour)
]
ax.legend(handles=handles, loc="lower right", fontsize=7,
          framealpha=0.85, ncol=2, markerscale=1.5)

plt.tight_layout()
plt.savefig(FIG_DIR / "spatial_cell_types.png", dpi=SPATIAL_DPI, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / 'spatial_cell_types.png'} (dpi={SPATIAL_DPI})")


Cell-type map: 300,000 random cells (of 926,006)
Saved → /home/steve/Projects/HeLab/BladderDIVE/output/figures/spatial_cell_types.png (dpi=300)


In [13]:
# QC spatial map — combined + one PNG per label (same limits → stackable / flip in a viewer)
qc_labels_series = adata.obs["qc_label"].astype(str)
qc_colour_map = {
    "Pass":               "steelblue",
    "Artifact_NoNucleus": "#d73027",
    "Artifact_Spillover": "#fc8d59",
}

if PLOT_QC_ALL_CELLS or n <= MAX_PLOT_CELLTYPE:
    xy_qc = xy
    qc_labs = qc_labels_series.to_numpy()
    print(f"QC maps: all {n:,} cells")
else:
    xy_qc = xy_plot_ct
    qc_labs = qc_labels_series.to_numpy()[idx_ct]
    print(f"QC maps: same subsample as cell-type map ({len(xy_qc):,} cells)")

qc_colours_arr = np.array([qc_colour_map.get(q, "#888888") for q in qc_labs])

def _save_qc_scatter(fname, x, y, c, title, s=0.45, alpha=0.75):
    fig, ax = plt.subplots(figsize=(14, 14))
    ax.scatter(x, y, c=c, s=s, alpha=alpha, linewidths=0, rasterized=True)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(title, fontsize=14)
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymax + pad, ymin - pad)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=SPATIAL_DPI, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIG_DIR / fname}")

# Combined overlay
_save_qc_scatter(
    "spatial_qc.png",
    xy_qc[:, 0], xy_qc[:, 1], qc_colours_arr,
    "Spatial map — QC labels (combined)",
)

# Separate layer per QC class (only points in that class; same canvas → "channels")
for lab, col in qc_colour_map.items():
    m = qc_labs == lab
    n_m = int(m.sum())
    if n_m == 0:
        continue
    pt_rgb = np.tile(mcolors.to_rgba(col)[:3], (m.sum(), 1))
    pt_rgba = np.column_stack([pt_rgb, np.full(m.sum(), 0.85)])
    _save_qc_scatter(
        f"spatial_qc_layer_{lab}.png",
        xy_qc[m, 0], xy_qc[m, 1], pt_rgba,
        f"QC only — {lab} (n={n_m:,})",
        s=0.55,
        alpha=1.0,
    )


QC maps: all 926,006 cells
Saved → /home/steve/Projects/HeLab/BladderDIVE/output/figures/spatial_qc.png
Saved → /home/steve/Projects/HeLab/BladderDIVE/output/figures/spatial_qc_layer_Pass.png
Saved → /home/steve/Projects/HeLab/BladderDIVE/output/figures/spatial_qc_layer_Artifact_NoNucleus.png
Saved → /home/steve/Projects/HeLab/BladderDIVE/output/figures/spatial_qc_layer_Artifact_Spillover.png


## 5. Marker Heatmap per Cell Type

Mean z-scored expression of each marker per cell type (clean cells only).
Each column = one marker, each row = one cell type.

**Reading the heatmap**: a row that is bright only in its defining markers is "clean".
A row that is bright everywhere suggests a threshold is still too low for that marker.


In [14]:
# Mean z-scored expression per cell type
from scipy.stats import zscore as _zscore

X_log = np.log1p(adata_clean.X)
ct_labels = adata_clean.obs["cell_type"].values
unique_cts = [ct for ct in adata.obs["cell_type"].cat.categories
              if ct not in ("Artifact_NoNucleus", "Artifact_Spillover")
              and (ct_labels == ct).sum() > 0]

mean_mat = np.vstack([X_log[ct_labels == ct].mean(axis=0) for ct in unique_cts])
mean_expr = pd.DataFrame(mean_mat, index=unique_cts, columns=adata_clean.var_names)

# Z-score each row (across markers) — use scipy directly on numpy array
mean_z = pd.DataFrame(
    _zscore(mean_expr.values, axis=1),
    index=mean_expr.index,
    columns=mean_expr.columns,
)

fig, ax = plt.subplots(figsize=(max(12, len(adata_clean.var_names)*0.6),
                                max(6,  len(unique_cts) * 0.4)))
sns.heatmap(
    mean_z,
    cmap="RdBu_r",
    center=0,
    vmin=-2, vmax=2,
    linewidths=0.3,
    linecolor="grey",
    ax=ax,
    cbar_kws={"label": "Mean z-scored log1p intensity"},
)
ax.set_title("Marker expression per cell type (z-scored)", fontsize=13)
ax.set_xlabel("Marker")
ax.set_ylabel("Cell type")
plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_cell_types.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Dot Plot

**Dot size** = percentage of cells in that cell type that are positive for the marker  
**Dot colour** = mean expression (log1p) among positive cells

This is the standard summary figure for cell-type marker validation in single-cell papers.


In [15]:
bin_layer = adata.layers["binary"]
marker_names = list(adata.var_names)
dapi_mask_all = ~np.array([m.upper().startswith("DAPI") for m in marker_names])
bin_no_dapi = bin_layer[:, dapi_mask_all]
X_no_dapi   = adata.X[:, dapi_mask_all]
markers_no_dapi = [m for m, keep in zip(marker_names, dapi_mask_all) if keep]

unique_cts_clean = [ct for ct in adata.obs["cell_type"].cat.categories
                    if ct not in ("Artifact_NoNucleus", "Artifact_Spillover")]

pct_pos  = np.zeros((len(unique_cts_clean), len(markers_no_dapi)))
mean_pos = np.zeros((len(unique_cts_clean), len(markers_no_dapi)))

ct_obs = adata.obs["cell_type"].values
for i, ct in enumerate(unique_cts_clean):
    mask = (ct_obs == ct) & adata.obs["is_clean"].values
    if mask.sum() == 0:
        continue
    pct_pos[i]  = 100 * bin_no_dapi[mask].mean(axis=0)
    pos_vals = np.log1p(X_no_dapi[mask])
    mean_pos[i] = pos_vals.mean(axis=0)

pct_df  = pd.DataFrame(pct_pos,  index=unique_cts_clean, columns=markers_no_dapi)
mean_df = pd.DataFrame(mean_pos, index=unique_cts_clean, columns=markers_no_dapi)

# Plot
n_ct  = len(unique_cts_clean)
n_mk  = len(markers_no_dapi)
fig, ax = plt.subplots(figsize=(n_mk * 0.55 + 1.5, n_ct * 0.45 + 1.5))

cmap = plt.get_cmap("YlOrRd")
max_expr = mean_df.values.max()

for i, ct in enumerate(unique_cts_clean):
    for j, mk in enumerate(markers_no_dapi):
        pct  = pct_df.loc[ct, mk]
        expr = mean_df.loc[ct, mk]
        if pct < 0.5:
            continue
        size = (pct / 100) ** 0.5 * 300
        colour = cmap(expr / max_expr if max_expr > 0 else 0)
        ax.scatter(j, i, s=size, color=colour, alpha=0.85, linewidths=0)

ax.set_xticks(range(n_mk))
ax.set_xticklabels(markers_no_dapi, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(n_ct))
ax.set_yticklabels(unique_cts_clean, fontsize=9)
ax.set_xlim(-0.8, n_mk - 0.2)
ax.set_ylim(-0.8, n_ct - 0.2)
ax.grid(True, alpha=0.25)
ax.set_title("Dot plot — % positive (size) × mean log1p expression (colour)", fontsize=11)

# Colourbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, max_expr))
plt.colorbar(sm, ax=ax, label="Mean log1p intensity", pad=0.01)

# Size legend
for pct_leg in [10, 25, 50, 75]:
    ax.scatter([], [], s=(pct_leg/100)**0.5*300, color="grey",
               label=f"{pct_leg}%", alpha=0.85)
ax.legend(title="% positive", loc="upper left", bbox_to_anchor=(1.12, 1),
          framealpha=0.85, fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "dotplot_cell_types.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Spatial Co-occurrence (Neighbourhood Enrichment)

For each pair of cell types, counts how often they appear as spatial neighbours
(within a radius `R` microns) vs what would be expected by chance.

Positive score → the two cell types are enriched near each other.  
Negative score → they avoid each other (spatial exclusion).

> Uses a manual implementation via `scipy.spatial.cKDTree`.
> Install `squidpy` for the full Ripley-K / nhood enrichment suite.


In [16]:
# Spatial co-occurrence — clean cells only, within radius R pixels
# (pixel size should be confirmed from metadata; default ~0.5 µm/px → R=100px ≈ 50 µm)
R = 100  # neighbour search radius in pixels

clean_mask = adata.obs["is_clean"].values
xy_clean   = adata.obsm["spatial"][clean_mask]
ct_clean   = adata.obs["cell_type"].values[clean_mask]

unique_cts_co = [ct for ct in np.unique(ct_clean)
                 if ct not in ("Unassigned",) and ct.startswith("A") == False]

# Build kd-tree and query neighbours
print(f"Building kd-tree for {len(xy_clean):,} cells ...")
tree = cKDTree(xy_clean)
pairs = tree.query_pairs(r=R, output_type="ndarray")
print(f"  Found {len(pairs):,} neighbour pairs (R={R}px)")

ct_to_idx = {ct: np.where(ct_clean == ct)[0] for ct in unique_cts_co}
n_cts = len(unique_cts_co)
obs_mat = np.zeros((n_cts, n_cts), dtype=float)
for a, b in pairs:
    ct_a, ct_b = ct_clean[a], ct_clean[b]
    if ct_a in ct_to_idx and ct_b in ct_to_idx:
        i = unique_cts_co.index(ct_a)
        j = unique_cts_co.index(ct_b)
        obs_mat[i, j] += 1
        if i != j:
            obs_mat[j, i] += 1

# Expected under random mixing (proportional to marginal counts)
marginal = obs_mat.sum(axis=1)
total    = obs_mat.sum()
exp_mat  = np.outer(marginal, marginal) / total if total > 0 else obs_mat
np.fill_diagonal(exp_mat, 0)

# Enrichment score: log2(observed / expected), clipped
with np.errstate(divide="ignore", invalid="ignore"):
    enrich = np.where(exp_mat > 0, np.log2(obs_mat / exp_mat), 0.0)
np.fill_diagonal(enrich, 0)

enrich_df = pd.DataFrame(enrich, index=unique_cts_co, columns=unique_cts_co)

fig, ax = plt.subplots(figsize=(max(8, n_cts * 0.55), max(6, n_cts * 0.45)))
sns.heatmap(
    enrich_df,
    cmap="RdBu_r",
    center=0,
    vmin=-2, vmax=2,
    linewidths=0.3,
    linecolor="lightgrey",
    ax=ax,
    annot=True, fmt=".1f", annot_kws={"fontsize": 7},
)
ax.set_title(f"Spatial co-occurrence (log2 enrichment, R={R}px)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "spatial_cooccurrence.png", dpi=150, bbox_inches="tight")
plt.show()


Building kd-tree for 617,128 cells ...
  Found 4,292,716 neighbour pairs (R=100px)


## 8. Save UMAP Coordinates

In [17]:
# Transfer UMAP back to the full adata for downstream use
adata.obsm["X_umap_clean"] = np.full((len(adata), 2), np.nan)
clean_idx = np.where(adata.obs["is_clean"].values)[0]
# Re-subset to same var selection
dapi_mask_full = ~np.array([m.upper().startswith("DAPI") for m in adata.var_names])
clean_idx_in_clean = np.arange(len(adata_clean))
adata.obsm["X_umap_clean"][clean_idx] = adata_clean.obsm["X_umap"]

adata.write_h5ad(CELLTYPES_H5AD)
print(f"Saved UMAP coordinates to {CELLTYPES_H5AD}")
print(f"  obsm key: X_umap_clean (NaN for artefact cells)")


Saved UMAP coordinates to /home/steve/Projects/HeLab/BladderDIVE/output/celldive_protein_matrix_celltypes.h5ad
  obsm key: X_umap_clean (NaN for artefact cells)


## Notes and next steps

### High Artifact_NoNucleus rate (~25%)
A 25% DAPI-negative rate is unusually high. Likely cause: the DAPI threshold is
currently set to the p25 of all DAPI intensities, which by definition flags 25%
of cells as negative. Consider:
- Re-running `determine_binary_threshold.ipynb` with DAPI included (not in `SKIP_MARKERS`)
  and using the GMM/Otsu threshold to separate real-nucleus vs background.
- Or use a fixed low absolute threshold (e.g. any cell with DAPI > 500 has a nucleus).

### Myofibroblast dominance (~37% of clean cells)
ACTA2 is positive in 35.6% of cells. In bladder cancer ACTA2 can be expressed in
smooth muscle, myofibroblasts, and activated fibroblasts. Confirm spatially whether the
ACTA2+ cells form peritumoral rings (expected) or are diffusely scattered (threshold issue).

### Refining thresholds further
After reviewing the spatial maps and UMAP, add any remaining problematic markers to
`MANUAL_OVERRIDES` in `determine_binary_threshold.ipynb` and re-run the pipeline.
